# Kill an running task

In [1]:
# This process is the same for all analytics

In [16]:
import base64
import json
from datetime import datetime
import requests
with open("token.txt", "r") as f:
    token = f.read().strip()
headers = {
    "Authorization": token
}

# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 3
# Select the organization IDs that can be selected by the user. These should be the IDs 
# of the vantage6 organizations:
#
#   1	- root
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#   8	- VGR
#   9   - OUS
#   10  - MSCI
#   11  - APHP
#
ORGANIZATION_IDS = [1, 4]
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 212
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 182
# The image to use is the latest version of the analytics algorithm
IMAGE = "ghcr.io/iknl/analytics:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm. For
# data exploration we use the `summary` method.
METHOD = "crosstab"
# The user is able to select multiple cohorts in RAVEN. These cohorts correspond to
# different dataframes, see the `2-new-cohort.ipynb` how these are created and check
# the `v6_dataframe` column in the RAVEN database for more information.
#
# In this example I've just used one cohort. Simply add more ids to the list to use
# more cohorts.
DATAFRAME_IDS = [474]
# Before we can start analysis the cohorts (dataframes) we need to check the variables
# that are available in the dataframes. You can use this endpoint whenever you want user
# to allow you to select variables.
# TODO the dtpyes might change in the future, so do not rely on them to heavily now. In
# the next version of the data extraction job we will likely provide you with either the
# `category` or `numeric` colum type (so that you can use them to select varables)
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/dataframe/{DATAFRAME_IDS[0]}",
    headers=headers
)
VARIABLES = response.json()["columns"]
VARIABLES

# The user is able to select two variables to compute the contingency table of.
RESULTS_COL = "sex"
# NOTE: This is where you add the group columns which can be one or more columns
GROUP_COLS = ["clinical_stage", "pathological_stage"]
org_input = [
    {
        "id": ORGANIZATION_IDS[0], # Central task
        "arguments": base64.b64encode(
            json.dumps(
                {
                    "results_col": RESULTS_COL,
                    "group_cols": GROUP_COLS,
                    # user selected participants
                    "organizations_to_include": ORGANIZATION_IDS 
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
]

payload = {
    "name": "Human-readable name of the task",
    "image": IMAGE,
    "description": "Description of the task",
    "action": "central_compute",
    "method": METHOD,
    "organizations": org_input,
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": df_id
            } for df_id in DATAFRAME_IDS
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}
payload
# Then using the authorization header and the payload we can create a vantage6 task
# using the vantage6 server API.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]

In [17]:
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/kill/task",
    headers=headers,
    json={"id": TASK_ID},
    timeout=30,
)

In [19]:
response.json()

{'msg': "Task 3112 already finished with status 'completed', so cannot kill it!"}

In [ ]:
# 200:
# {'msg': 'Nodes have been instructed to kill any containers running for task 3111.'}
# 400:
# {'msg': "Task 3112 already finished with status 'completed', so cannot kill it!"}
